In [0]:
import yaml
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%run ../../adb_local/Utilities/de_utilities

In [0]:
# yaml_path = "/Workspace/Users/shivec195@gmail.com/adb/config/schema_definition/bronze_schema/loan_defaulters.yml"
# file_path = "/Volumes/dltshiv/source/files/loan_defaulters/"

In [0]:
# read the Yaml file and store into variable
with open(yaml_path, "r") as file:
    yaml_file = yaml.safe_load(file)

#retrieving the variables from YAML file
schema_name =yaml_file.get("schema_name")
table_name  =yaml_file.get("table_name")
file_name   =yaml_file.get("file_name")
column_list =yaml_file.get("columns",[])

# create a custom schema from YAML file
modify_schema = StructType(
    [
        StructField(column["source_column"],StringType(),True) for column in column_list
    ]
)

#Reading the CSV file and loading into the dataframe 
df_default_loan = spark.read.format("csv").option("header","true").schema(modify_schema).load(path+file_name)

In [0]:
# creating column mapping for destination columns
column_mapping = [(column["source_column"],column["destination_column"]) for column in column_list]

In [0]:
#renameing dataframe columns as per the destination columns
df_default_loan_table = df_default_loan.select([col(column[0]).alias(column[1]) for column in column_mapping])

#preparing final table name
final_table_name = "bronze."+schema_name+"."+table_name

#writing the dataframe to the table
df_default_loan_table.write.mode("append").insertInto(final_table_name)
# df_default_loan_table.write.mode("overwrite").saveAsTable(final_table_name)

In [0]:
%sql
select * from bronze.bronze_schema.loan_defaulters